# Förster Resonance Energy Transfer (FRET)

<a href="https://colab.research.google.com/github/elkins/diff-fret/blob/main/examples/interactive_tutorials/fret_efficiency_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Förster Resonance Energy Transfer (FRET) is a non-radiative energy transfer mechanism between two light-sensitive molecules (fluorophores). Because the transfer efficiency is strongly dependent on distance, FRET is widely used as a "spectroscopic ruler" to measure nanometer-scale distances in proteins and nucleic acids.

According to Förster theory (1948), the transfer efficiency $E$ between a donor and acceptor separated by distance $r$ is:
$$ E = \frac{1}{1 + (r/R_0)^6} $$

where $R_0$ is the Förster distance (the distance at which transfer efficiency is 50%). $R_0$ depends on the overlap integral of the donor emission and acceptor absorption spectra, the quantum yield of the donor, the refractive index, and the relative orientation of the transition dipoles ($\kappa^2$).

This tutorial demonstrates how to simulate FRET efficiencies using the `diff_fret` physics module.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/elkins/diff-fret.git diff-fret matplotlib biotite
else:
    sys.path.append("../../")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

## 1. The Förster Curve

Let's visualize the standard Förster efficiency curve using the `fret_efficiency` function. We'll use a typical $R_0$ value of 50 Å.

In [ ]:
from diff_fret.kernels import fret_efficiency

r_values = jnp.linspace(10.0, 100.0, 200)
R0 = 50.0

efficiency = fret_efficiency(r_values, r0=R0)

plt.figure(figsize=(8, 5))
plt.plot(np.array(r_values), np.array(efficiency), linewidth=3, color="darkorange")
plt.axvline(R0, color="gray", linestyle="--", label=f"$R_0$ = {R0} Å")
plt.axhline(0.5, color="gray", linestyle="--")
plt.xlabel("Distance $r$ (Å)", fontsize=12)
plt.ylabel("FRET Efficiency $E$", fontsize=12)
plt.title("Förster Resonance Energy Transfer Efficiency", fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 2. Accessible Volume (AV) Simulation

In realistic structural biology applications, fluorescent dyes are attached to the protein backbone via flexible linkers. The dye can sample a cloud of positions, known as the Accessible Volume (AV). 

Using the simple distance between the $C_\alpha$ atoms will systematically misestimate the FRET efficiency. We must average the efficiency over the dye position distributions:
$$ \langle E \rangle = \int \int P_D(\vec{r}_D) P_A(\vec{r}_A) \frac{1}{1 + (|\vec{r}_D - \vec{r}_A| / R_0)^6} d\vec{r}_D d\vec{r}_A $$

The `fret_efficiency_av` function performs a differentiable Monte Carlo integration of this Accessible Volume using Gaussian clouds.

In [ ]:
from diff_fret.kernels import fret_efficiency_av

# Define attachment points (e.g., C-alpha coordinates)
attachment_donor = jnp.array([0.0, 0.0, 0.0])
attachment_acceptor = jnp.array([45.0, 0.0, 0.0])

# Estimate <E> using Accessible Volume simulation
avg_e = fret_efficiency_av(
    attachment_donor=attachment_donor,
    attachment_acceptor=attachment_acceptor,
    radius_donor=8.0,  # Dye linker flexibility
    radius_acceptor=8.0,
    n_samples=5000,  # Number of MC samples
    r0=50.0,
    key=jax.random.PRNGKey(42),
)

print("Distance between attachment points: 45.0 Å")
print(f"Point-to-point efficiency: {fret_efficiency(jnp.array(45.0), 50.0):.4f}")
print(f"Accessible Volume <E>: {avg_e:.4f}")

### Visualizing the Dye Position Clouds

Let's use Matplotlib to visualize the Gaussian clouds that `fret_efficiency_av` samples internally.

In [ ]:
# Generate samples to visualize
key_d, key_a = jax.random.split(jax.random.PRNGKey(42))
samples_d = attachment_donor + 8.0 * jax.random.normal(key_d, (1000, 3))
samples_a = attachment_acceptor + 8.0 * jax.random.normal(key_a, (1000, 3))

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection="3d")

# Plot the attachment points
ax.scatter(*attachment_donor, color="blue", s=100, marker="*", label="Donor Attachment")
ax.scatter(*attachment_acceptor, color="red", s=100, marker="*", label="Acceptor Attachment")

# Plot the dye clouds
ax.scatter(
    samples_d[:, 0],
    samples_d[:, 1],
    samples_d[:, 2],
    color="blue",
    alpha=0.05,
    s=10,
    label="Donor AV",
)
ax.scatter(
    samples_a[:, 0],
    samples_a[:, 1],
    samples_a[:, 2],
    color="red",
    alpha=0.05,
    s=10,
    label="Acceptor AV",
)

# Draw the backbone pseudo-bond
ax.plot(
    [attachment_donor[0], attachment_acceptor[0]],
    [attachment_donor[1], attachment_acceptor[1]],
    [attachment_donor[2], attachment_acceptor[2]],
    color="black",
    linestyle="--",
    label="Backbone distance",
)

ax.set_xlabel("X (Å)")
ax.set_ylabel("Y (Å)")
ax.set_zlabel("Z (Å)")
ax.set_title("Flexible Dye Accessible Volumes (AV)")

# Fix legend alpha for visibility
leg = ax.legend(loc="upper right")
for lh in leg.legend_handles:
    lh.set_alpha(1)

plt.tight_layout()
plt.show()

## 3. Orientation Factor ($\kappa^2$) Limits

The Förster distance $R_0$ depends on $\kappa^2$, the relative orientation of the transition dipoles. It is usually assumed to be 2/3 (dynamic isotropic averaging). However, if the dyes are restricted in their mobility, $\kappa^2$ can vary between 0 and 4.

The Dale-Eisinger-Blumberg (1979) theory allows us to place bounds on $\kappa^2$ based on the steady-state fluorescence anisotropy of the dyes.

In [ ]:
from diff_fret.kernels import kappa_squared_bounds

# Measured steady-state anisotropies
anisotropy_donor = 0.15
anisotropy_acceptor = 0.20

bounds = kappa_squared_bounds(anisotropy_donor, anisotropy_acceptor)

print(f"Donor Anisotropy:    {anisotropy_donor}")
print(f"Acceptor Anisotropy: {anisotropy_acceptor}")
print(f"Kappa^2 minimum:     {bounds[0]:.4f}")
print(f"Kappa^2 maximum:     {bounds[1]:.4f}")
print("(Isotropic limit is 2/3 ≈ 0.6667)")